# Setup
Project paths and constants.

In [ ]:
import os
import json
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import mobilenet_v2
import matplotlib.pyplot as plt

from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

In [ ]:
NOTEBOOK_PATH = Path.cwd()
if NOTEBOOK_PATH.name == "notebooks":
    PROJECT_ROOT = NOTEBOOK_PATH.parent
else:
    PROJECT_ROOT = NOTEBOOK_PATH

DATA_DIR = PROJECT_ROOT / "data" / "new_plant_diseases_dataset"
VALID_DIR = DATA_DIR / "valid"
TEST_DIR = DATA_DIR / "test"

MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = RESULTS_DIR / "figures"

MOBILENET_PATH = MODELS_DIR / "crop_disease_mobilenetv2_1.keras"
EFFICIENTNET_PATH = MODELS_DIR / "crop_disease_efficientnetb0_2.keras"

COMPARISON_JSON_PATH = RESULTS_DIR / "model_comparison.json"
COMPARISON_TABLE_PATH = RESULTS_DIR / "model_comparison_table.csv"

COMPARISON_ACCURACY_FIG_PATH = FIGURES_DIR / "model_comparison_accuracy.png"
COMPARISON_LOSS_FIG_PATH = FIGURES_DIR / "model_comparison_loss.png"

BEST_CLASSIFICATION_REPORT_PATH = RESULTS_DIR / "best_model_classification_report.csv"
BEST_METRICS_PATH = RESULTS_DIR / "best_model_metrics.json"

BEST_CONFUSION_MATRIX_PATH = FIGURES_DIR / "best_model_confusion_matrix.png"
BEST_SAMPLE_PREDICTIONS_PATH = FIGURES_DIR / "best_model_sample_predictions.png"
BEST_GRADCAM_PATH = FIGURES_DIR / "best_model_gradcam_examples.png"

IMG_SIZE = (224, 224)
IMAGE_SIZE = IMG_SIZE
BATCH_SIZE = 32
SEED = 30

print("PROJECT_ROOT:", PROJECT_ROOT)
print("VALID_DIR:", VALID_DIR)
print("TEST_DIR:", TEST_DIR)


# Load Dataset
Use `valid/` for model comparison, selection, and labeled evaluation outputs. Use `test/` only for unlabeled demo predictions and Grad-CAM examples.


In [ ]:
raw_valid_ds = tf.keras.utils.image_dataset_from_directory(
    VALID_DIR,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

class_names = raw_valid_ds.class_names
num_classes = len(class_names)
AUTOTUNE = tf.data.AUTOTUNE

test_image_paths = [
    os.path.join(TEST_DIR, fname)
    for fname in sorted(os.listdir(TEST_DIR))
    if fname.lower().endswith((".jpg", ".jpeg", ".png"))
]


def load_test_image(path):
    image = tf.io.read_file(path)
    image = tf.image.decode_image(image, channels=3, expand_animations=False)
    image = tf.image.resize(image, IMAGE_SIZE)
    image = tf.cast(image, tf.uint8)
    return image, path


raw_test_ds = tf.data.Dataset.from_tensor_slices(test_image_paths)
raw_test_ds = raw_test_ds.map(load_test_image, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)


def prepare_dataset(model_name, dataset=None):
    if dataset is None:
        dataset = raw_valid_ds

    if model_name == "MobileNetV2":
        return dataset.map(
            lambda images, labels: (
                mobilenet_v2.preprocess_input(tf.cast(images, tf.float32)),
                labels,
            ),
            num_parallel_calls=AUTOTUNE,
        ).prefetch(AUTOTUNE)

    return dataset.prefetch(AUTOTUNE)


print(f"Classes: {num_classes}")
print(f"Validation batches: {tf.data.experimental.cardinality(raw_valid_ds).numpy()}")
print(f"Unlabeled test images: {len(test_image_paths)}")
print(f"Test batches: {tf.data.experimental.cardinality(raw_test_ds).numpy()}")


# Load Models
Load the two trained `.keras` files.

In [ ]:
mobilenet_model = keras.models.load_model(MOBILENET_PATH)
efficientnet_model = keras.models.load_model(EFFICIENTNET_PATH)

print("Loaded MobileNetV2:", MOBILENET_PATH)
print("Loaded EfficientNetB0:", EFFICIENTNET_PATH)

# Compare Models
Evaluate both models on `valid/`.

In [ ]:
comparison_rows = []

for model_name, model_path, model in [
    ("MobileNetV2", MOBILENET_PATH, mobilenet_model),
    ("EfficientNetB0", EFFICIENTNET_PATH, efficientnet_model),
]:
    print("\nEvaluating:", model_name)
    print("Model path:", model_path)

    eval_ds = prepare_dataset(model_name)
    val_loss, val_accuracy = model.evaluate(eval_ds, verbose=1)

    comparison_rows.append({
        "model": model_name,
        "model_path": str(model_path),
        "validation_accuracy": float(val_accuracy),
        "validation_loss": float(val_loss),
    })

comparison_df = pd.DataFrame(comparison_rows).sort_values(
    by=["validation_accuracy", "validation_loss"],
    ascending=[False, True],
).reset_index(drop=True)

comparison_df


In [ ]:
comparison_results = {
    row["model"]: {
        "validation_accuracy": float(row["validation_accuracy"]),
        "validation_loss": float(row["validation_loss"]),
        "model_path": str(row["model_path"]),
        "evaluation_split": "valid folder"
    }
    for row in comparison_rows
}

best_row = comparison_df.iloc[0]

comparison_results["best_model"] = {
    "model": best_row["model"],
    "validation_accuracy": float(best_row["validation_accuracy"]),
    "validation_loss": float(best_row["validation_loss"]),
    "model_path": str(best_row["model_path"])
}

comparison_results["note"] = (
    "Both models were evaluated on the same labeled validation split. "
    "MobileNetV2 uses its standard preprocess_input function. "
    "EfficientNetB0 uses the raw image pipeline from the dataset loader."
)

with open(COMPARISON_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(comparison_results, f, indent=4)

comparison_df.to_csv(COMPARISON_TABLE_PATH, index=False)

print("Saved:", COMPARISON_JSON_PATH)
print("Saved:", COMPARISON_TABLE_PATH)

comparison_results


In [ ]:
plt.figure(figsize=(7, 4.5))
plt.bar(comparison_df["model"], comparison_df["validation_accuracy"] * 100)
plt.title("Model Comparison - Validation Accuracy", fontsize=13, fontweight="bold")
plt.xlabel("Model")
plt.ylabel("Validation Accuracy (%)")
plt.grid(axis="y", linestyle="--", alpha=0.5)

for i, value in enumerate(comparison_df["validation_accuracy"] * 100):
    plt.text(i, value + 0.4, f"{value:.2f}%", ha="center", fontsize=10)

plt.tight_layout()
plt.savefig(COMPARISON_ACCURACY_FIG_PATH, dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(7, 4.5))
plt.bar(comparison_df["model"], comparison_df["validation_loss"])
plt.title("Model Comparison - Validation Loss", fontsize=13, fontweight="bold")
plt.xlabel("Model")
plt.ylabel("Validation Loss")
plt.grid(axis="y", linestyle="--", alpha=0.5)

for i, value in enumerate(comparison_df["validation_loss"]):
    plt.text(i, value + 0.005, f"{value:.4f}", ha="center", fontsize=10)

plt.tight_layout()
plt.savefig(COMPARISON_LOSS_FIG_PATH, dpi=300, bbox_inches="tight")
plt.show()

# Best Model
Use the model with higher validation accuracy.

In [ ]:
best_model_name = comparison_df.iloc[0]["model"]
best_model_path = comparison_df.iloc[0]["model_path"]

if best_model_name == "MobileNetV2":
    best_model = mobilenet_model
else:
    best_model = efficientnet_model

print("Best model:", best_model_name)
print("Best model path:", best_model_path)

In [ ]:
best_eval_ds = prepare_dataset(best_model_name, raw_valid_ds)

y_true = []
y_pred = []

for images, labels in best_eval_ds:
    predictions = best_model.predict(images, verbose=0)
    predicted_classes = np.argmax(predictions, axis=1)

    y_true.extend(labels.numpy())
    y_pred.extend(predicted_classes)

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print("Validation samples:", len(y_true))


# Classification Report
Predict on `valid/` and save the labeled evaluation report for the selected best model.


In [ ]:
report = classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    output_dict=True
)

report_df = pd.DataFrame(report).transpose()
report_df.to_csv(BEST_CLASSIFICATION_REPORT_PATH)

print("Saved:", BEST_CLASSIFICATION_REPORT_PATH)
report_df.head()


# Confusion Matrix
Save the validation confusion matrix for the selected best model.


In [ ]:
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(18, 18))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names
)

disp.plot(
    ax=ax,
    xticks_rotation=90,
    cmap="Blues",
    colorbar=True,
    values_format="d"
)

plt.title(f"Confusion Matrix - {best_model_name}", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(BEST_CONFUSION_MATRIX_PATH, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", BEST_CONFUSION_MATRIX_PATH)


# Sample Predictions
Show 16 unlabeled test images with predicted labels and confidence.


In [ ]:
def clean_label(label):
    if "___" in label:
        crop, condition = label.split("___", maxsplit=1)
    else:
        crop, condition = label, "Unknown"

    crop = crop.replace("_", " ").strip()
    condition = condition.replace("_", " ").strip()

    return crop, condition

In [ ]:
plt.figure(figsize=(14, 14))

for images, paths in raw_test_ds.take(1):
    if best_model_name == "MobileNetV2":
        model_inputs = mobilenet_v2.preprocess_input(tf.cast(images, tf.float32))
    else:
        model_inputs = images

    predictions = best_model.predict(model_inputs, verbose=0)
    predicted_classes = np.argmax(predictions, axis=1)
    confidence_scores = np.max(predictions, axis=1)

    for i in range(min(16, len(images))):
        img = images[i].numpy().astype("uint8")

        predicted_label = class_names[int(predicted_classes[i])]
        confidence = confidence_scores[i] * 100
        pred_crop, pred_condition = clean_label(predicted_label)
        filename = Path(paths[i].numpy().decode("utf-8")).name

        plt.subplot(4, 4, i + 1)
        plt.imshow(img)
        plt.axis("off")
        plt.title(
            f"{filename}\n"
            f"P: {pred_crop}\n{pred_condition}\n"
            f"{confidence:.1f}%",
            fontsize=8
        )

plt.tight_layout()
plt.savefig(BEST_SAMPLE_PREDICTIONS_PATH, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", BEST_SAMPLE_PREDICTIONS_PATH)


In [ ]:
best_metrics = {
    "best_model": best_model_name,
    "model_path": str(best_model_path),
    "selection_split": "valid folder",
    "validation_accuracy": float(comparison_df.iloc[0]["validation_accuracy"]),
    "validation_loss": float(comparison_df.iloc[0]["validation_loss"]),
    "num_classes": int(num_classes),
    "evaluation_split": "valid folder",
    "demo_split": "test folder"
}

with open(BEST_METRICS_PATH, "w", encoding="utf-8") as f:
    json.dump(best_metrics, f, indent=4)

print("Saved:", BEST_METRICS_PATH)
best_metrics


# Grad-CAM
Grad-CAM highlights unlabeled test-set image regions that influenced the selected best model predictions.

In [ ]:
def make_gradcam_heatmap(img_array, model, base_model_index, last_conv_layer_name, pred_index=None):
    base_model = model.layers[base_model_index]
    conv_output_model = keras.Model(
        base_model.input,
        [base_model.get_layer(last_conv_layer_name).output, base_model.output],
    )

    with tf.GradientTape() as tape:
        x = img_array
        for layer in model.layers[:base_model_index]:
            x = layer(x, training=False)

        conv_outputs, x = conv_output_model(x, training=False)

        for layer in model.layers[base_model_index + 1:]:
            x = layer(x, training=False)

        predictions = x
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = tf.reduce_sum(conv_outputs[0] * pooled_grads, axis=-1)
    heatmap = tf.maximum(heatmap, 0)

    max_value = tf.reduce_max(heatmap)
    if float(max_value) > 0:
        heatmap /= max_value

    return heatmap.numpy()


def overlay_gradcam(raw_image, heatmap, alpha=0.25):
    heatmap = np.uint8(255 * heatmap)
    colors = plt.get_cmap("jet")(np.arange(256))[:, :3]
    heatmap = colors[heatmap]

    heatmap = keras.utils.array_to_img(heatmap)
    heatmap = heatmap.resize((raw_image.shape[1], raw_image.shape[0]))
    heatmap = keras.utils.img_to_array(heatmap)

    overlay = heatmap * alpha + raw_image.astype("float32")
    return np.clip(overlay / 255.0, 0.0, 1.0)


In [ ]:
gradcam_model = best_model

sample_batch = next(iter(raw_test_ds))
sample_image = np.expand_dims(sample_batch[0][0].numpy(), axis=0)

if best_model_name == "MobileNetV2":
    sample_image_for_model = mobilenet_v2.preprocess_input(tf.cast(sample_image, tf.float32))
else:
    sample_image_for_model = sample_image

_ = gradcam_model(sample_image_for_model, training=False)

base_model_index = next(
    i for i in range(len(gradcam_model.layers) - 1, -1, -1)
    if isinstance(gradcam_model.layers[i], keras.Model)
)
base_model = gradcam_model.layers[base_model_index]

conv_layer_types = (layers.Conv2D, layers.DepthwiseConv2D, layers.SeparableConv2D)
last_conv_layer_name = next(
    layer.name for layer in reversed(base_model.layers)
    if isinstance(layer, conv_layer_types)
)

print("Grad-CAM model:", best_model_name)
print("Base model:", base_model.name)
print("Last conv layer:", last_conv_layer_name)


In [ ]:
gradcam_images = []
gradcam_titles = []

for images, paths in raw_test_ds:
    for i in range(images.shape[0]):
        raw_image = images[i].numpy().astype("uint8")
        model_input = np.expand_dims(raw_image, axis=0)

        if best_model_name == "MobileNetV2":
            model_input_for_model = mobilenet_v2.preprocess_input(tf.cast(model_input, tf.float32))
        else:
            model_input_for_model = model_input

        pred_probs = gradcam_model.predict(model_input_for_model, verbose=0)
        pred_id = int(np.argmax(pred_probs[0]))
        confidence = float(np.max(pred_probs[0]) * 100)

        heatmap = make_gradcam_heatmap(
            model_input_for_model,
            gradcam_model,
            base_model_index,
            last_conv_layer_name,
            pred_index=pred_id,
        )

        pred_label = class_names[pred_id]
        filename = Path(paths[i].numpy().decode("utf-8")).name

        gradcam_images.append(overlay_gradcam(raw_image, heatmap))
        gradcam_titles.append(f"{filename}\nP: {pred_label}\nConf: {confidence:.1f}%")

        if len(gradcam_images) == 9:
            break

    if len(gradcam_images) == 9:
        break

print(f"Collected {len(gradcam_images)} Grad-CAM samples")


In [ ]:
num_examples = len(gradcam_images)
num_to_show = min(9, num_examples)

fig, axes = plt.subplots(3, 3, figsize=(14, 14))
axes = axes.flatten()

for i, ax in enumerate(axes):
    if i < num_to_show:
        ax.imshow(gradcam_images[i])
        ax.set_title(gradcam_titles[i], fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.savefig(BEST_GRADCAM_PATH, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", BEST_GRADCAM_PATH)


# Short Summary
MobileNetV2 and EfficientNetB0 are compared on `valid/`, and the best model is selected from the validation results in this notebook.

The selected best model is evaluated on `valid/` for the classification report and confusion matrix, while unlabeled `test/` images are used only for sample predictions and Grad-CAM visualizations.


In [ ]:
print("Model comparison complete.")
print("Best model:", best_model_name)
print("Validation accuracy:", comparison_df.iloc[0]["validation_accuracy"])
print("Validation loss:", comparison_df.iloc[0]["validation_loss"])
print("Saved comparison:", COMPARISON_JSON_PATH)
print("Saved report:", BEST_CLASSIFICATION_REPORT_PATH)
print("Saved confusion matrix:", BEST_CONFUSION_MATRIX_PATH)
print("Saved sample predictions:", BEST_SAMPLE_PREDICTIONS_PATH)
print("Saved Grad-CAM:", BEST_GRADCAM_PATH)
